# Practicum 2 en 3: fouten per groep en uitlegbaarheid

AI & Ethics (A-ET), blok 1. Dit notebook doet dezelfde stappen als de Orange-versie in kennispakket 1. Deel 1 is practicum 2 (het bias-experiment), deel 2 is practicum 3 (uitlegbaarheid).

Je hebt drie bestanden nodig, in dezelfde map als dit notebook: `suribank_versie_A.csv`, `suribank_versie_B.csv` en `suribank_testset.csv`.

**Het verhaal achter versie B.** Tot 2022 werden betalingen van klanten uit het binnenland via het kantoor in Albina met de hand ingeboekt. Een deel van de klanten die gewoon betaalden, staat daardoor in de historische data als wanbetaler. Versie B bevat die scheve labels. De testset bevat de echte uitkomsten.

**Let op:** Python rekent op een paar punten anders dan Orange. Je getallen kunnen iets afwijken. Het patroon bij practicum 2 is hetzelfde. Bij practicum 3 scoort logistische regressie in Python hoger dan in Orange; de les blijft: de lange boom doet het slechter dan de korte, en de korte boom is bijna even goed en wel te begrijpen.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.dummy import DummyClassifier

DOEL = "terugbetaald"   # het label: betaalde de aanvrager terug?
ID = "aanvraag_id"      # een nummer zegt niets over de aanvrager, dus dit gebruiken we niet

def voorbewerking(kenmerken, data, schalen):
    tekst = [k for k in kenmerken if data[k].dtype == object]
    getal = [k for k in kenmerken if k not in tekst]
    stappen = [SimpleImputer(strategy="median")] + ([StandardScaler()] if schalen else [])
    return ColumnTransformer([
        ("tekst", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), tekst),
        ("getal", make_pipeline(*stappen), getal)], verbose_feature_names_out=False)

def logistische_regressie(kenmerken, data):
    return Pipeline([("prep", voorbewerking(kenmerken, data, schalen=True)), ("model", LogisticRegression(max_iter=1000))])

def boom(kenmerken, data, diepte=None):
    return Pipeline([("prep", voorbewerking(kenmerken, data, schalen=False)),
                     ("model", DecisionTreeClassifier(max_depth=diepte, min_samples_leaf=2, min_samples_split=5, random_state=0))])

def kans_ja(model, data, kenmerken):
    ja = list(model.classes_).index("ja")
    return model.predict_proba(data[kenmerken])[:, ja]

versie_A = pd.read_csv("suribank_versie_A.csv")
versie_B = pd.read_csv("suribank_versie_B.csv")
testset = pd.read_csv("suribank_testset.csv")
alle_kenmerken = [k for k in versie_A.columns if k not in (DOEL, ID)]
GEVOELIG = ["geslacht", "district", "regio"]
print(len(versie_A), "aanvragen om te leren,", len(testset), "om te testen")

## Deel 1. Practicum 2: het bias-experiment

We leren logistische regressie op de leerdata en testen apart voor de kust en het binnenland. Per regio rekenen we uit:

- **Goedgekeurd:** voorspeld ja, gedeeld door het totaal.
- **Onterecht afgewezen:** goede betalers met voorspelling nee, gedeeld door alle goede betalers.
- **Onterecht goedgekeurd:** slechte betalers met voorspelling ja, gedeeld door alle slechte betalers.

In [ ]:
def per_regio(leerdata, kenmerken, naam):
    m = logistische_regressie(kenmerken, leerdata).fit(leerdata[kenmerken], leerdata[DOEL])
    rijen = []
    for regio in ["kust", "binnenland"]:
        t = testset[testset["regio"] == regio]
        v = m.predict(t[kenmerken]); e = t[DOEL].values
        rijen.append({"versie": naam, "regio": regio,
                      "CA": round((v == e).mean() * 100, 1),
                      "goedgekeurd": round((v == "ja").mean() * 100, 1),
                      "onterecht afgewezen": round(((v == "nee") & (e == "ja")).sum() / (e == "ja").sum() * 100, 1),
                      "onterecht goedgekeurd": round(((v == "ja") & (e == "nee")).sum() / (e == "nee").sum() * 100, 1)})
    return rijen

### Stap 1 tot en met 4. Versie A

In [ ]:
tabel = per_regio(versie_A, alle_kenmerken, "A")
pd.DataFrame(tabel)

### Stap 5. Versie B, met de scheve labels

In [ ]:
tabel += per_regio(versie_B, alle_kenmerken, "B")
pd.DataFrame(tabel)

### Stap 6. Versie B zonder gevoelige kenmerken

Geslacht, district en regio gaan eruit, net als met Select Columns in Orange.

In [ ]:
zonder = [k for k in alle_kenmerken if k not in GEVOELIG]
tabel += per_regio(versie_B, zonder, "B zonder gevoelige kenmerken")
uitkomst2 = pd.DataFrame(tabel)
uitkomst2

### Vragen bij practicum 2

1. Versie A: hoe groot is het verschil in onterecht afwijzen tussen kust en binnenland? Waar komt dat verschil vandaan, als woonplaats in deze data geen directe rol speelt bij terugbetalen?
2. Versie B: wat gebeurt er met goede betalers uit het binnenland? Wat gebeurt er met de CA voor het binnenland? Hoe kan dat?
3. Zonder gevoelige kenmerken: wat verandert er? Waarom kom je niet terug op het niveau van versie A?
4. Wie is verantwoordelijk voor het verschil: de bank die de data vastlegde, de leverancier die alle velden gebruikte, of de bank die niet per groep testte? Gebruik minstens één ethische lens.
5. Wat zou jij van Suribank eisen voordat zo'n model live mag? Formuleer één eis die je kunt controleren.

Vul daarna evidence card 2 in.

## Deel 2. Practicum 3: uitlegbaarheid

### Stap 1 en 2. Drie modellen vergelijken

We leren op versie A, zonder geslacht, district en regio: een korte boom (3 niveaus), een lange boom en logistische regressie.

In [ ]:
k = [x for x in alle_kenmerken if x not in GEVOELIG]
modellen3 = {
    "Korte boom (3 niveaus)": boom(k, versie_A, diepte=3),
    "Lange boom": boom(k, versie_A),
    "Logistic Regression": logistische_regressie(k, versie_A),
}
rijen = []
for naam, m in modellen3.items():
    m.fit(versie_A[k], versie_A[DOEL])
    ca = (m.predict(testset[k]) == testset[DOEL]).mean() * 100
    rijen.append({"model": naam, "CA op de testset": round(ca, 1),
                  "eindpunten": m.named_steps["model"].get_n_leaves() if "boom" in naam else None})
pd.DataFrame(rijen).set_index("model")

### Stap 3. In de korte boom kijken

Lees de regels van boven naar beneden. `class: ja` betekent: het model voorspelt terugbetalen.

In [ ]:
korte = modellen3["Korte boom (3 niveaus)"]
print(export_text(korte.named_steps["model"], feature_names=list(korte.named_steps["prep"].get_feature_names_out())))

### Stap 4. In logistische regressie kijken

In Orange gebruik je hiervoor het Nomogram. Hier kijken we naar de gewichten. Positief betekent: meer kans op ja. Omdat de getallen geschaald zijn, kun je de gewichten met elkaar vergelijken.

In [ ]:
lr3 = modellen3["Logistic Regression"]
namen = lr3.named_steps["prep"].get_feature_names_out()
richting = -1 if lr3.classes_[1] == "nee" else 1   # scikit-learn rekent richting de tweede klasse
gewichten = pd.Series(richting * lr3.named_steps["model"].coef_[0], index=namen)
gewichten.sort_values(key=abs, ascending=False).round(3).head(10)

### Stap 5. Eén aanvrager: TB0046

Wat voorspelt elk model voor deze aanvrager, met welke kans, en wat gebeurde er werkelijk?

In [ ]:
aanvrager = testset[testset[ID] == "TB0046"]
print(aanvrager[k + [DOEL]].T)
print()
for naam, m in modellen3.items():
    p = kans_ja(m, aanvrager, k)[0]
    print(f"{naam:24s} voorspelt {'ja' if p >= 0.5 else 'nee'}, kans op ja {p:.0%}")
print("Werkelijk terugbetaald:", aanvrager[DOEL].iloc[0])

### Vragen bij practicum 3

1. Welke CA hebben de drie modellen op de testset? Welk model is het best? Verrast je dat?
2. Schrijf de korte boom op als drie of vier regels in gewone taal, zoals: "Bij een looptijd van meer dan 24 maanden en geen eerdere achterstand …"
3. Welke drie kenmerken wegen bij logistische regressie het zwaarst? Vind je dat redelijk?
4. Wat voorspelt elk model voor aanvraag TB0046, en met welke kans? Wat gebeurde er werkelijk?
5. Leg de uitkomst van de korte boom voor TB0046 uit aan drie doelgroepen, elk in hooguit 60 woorden: de aanvrager, de kredietmedewerker en een toezichthouder.
6. Rudin zegt: kies bij grote gevolgen een model dat zelf te begrijpen is. Ben je het daarmee eens voor Suribank? Wanneer zou je toch een complexer model kiezen?

Vul daarna evidence card 3 in.